# sc4000 - training (internet ON)
Trains the DINOv3 pair classifier head and gathers everything the **offline** inference notebook needs.
The output of this notebook is mounted in `kaggle/inference` via `kernel_sources` as `/kaggle/input/sc4000-training/`:

| output path | used by |
|---|---|
| `dinov3-vitb16-pretrain-lvd1689m/` | step 1 backbone (`--dinov3-model-dir`) |
| `pair_classifier/pair_classifier_head_dinov3.pt` | step 1 head (`--pair-classifier-checkpoint-path`) |
| `wheels/` | transformers >= 4.56 (DINOv3) for the offline image |
| `aliked_lightglue_weights/` | step 4 ALIKED + LightGlue weights |

Setup: add a Kaggle secret `HF_TOKEN` with access to the gated `facebook/dinov3-vitb16-pretrain-lvd1689m`,
and upload this repo's `src/` + `scripts/` as the dataset `sc4000-src`.

In [ ]:
!pip download --no-deps -d /kaggle/working/wheels "transformers>=4.56,<5" "huggingface_hub>=0.34,<1.0" "tokenizers>=0.22,<0.23" "safetensors>=0.4.3"
!pip install --no-deps /kaggle/working/wheels/*.whl

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

snapshot_download(
    "facebook/dinov3-vitb16-pretrain-lvd1689m",
    local_dir="/kaggle/working/dinov3-vitb16-pretrain-lvd1689m",
    token=UserSecretsClient().get_secret("HF_TOKEN"),
)

In [ ]:
import os, torch
weights_dir = "/kaggle/working/aliked_lightglue_weights"
os.makedirs(weights_dir, exist_ok=True)
torch.hub.download_url_to_file("https://github.com/Shiaoming/ALIKED/raw/main/models/aliked-n16.pth", f"{weights_dir}/aliked-n16.pth")
torch.hub.download_url_to_file("https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/aliked_lightglue.pth", f"{weights_dir}/aliked_lightglue.pth")

In [ ]:
!python /kaggle/input/sc4000-src/scripts/train.py \
    --competition-data-dir /kaggle/input/image-matching-challenge-2025 \
    --dinov3-model-dir /kaggle/working/dinov3-vitb16-pretrain-lvd1689m \
    --output-dir /kaggle/working/pair_classifier \
    --val-datasets imc2023_haiper imc2024_lizard_pond \
    --epochs 30